# Worked Capstone: House Price Regression

**Domain:** Supervised regression  
**Primary dataset:** `house_prices.csv`  
**Level:** Practitioner to Advanced

## Business goal

Predict house prices for analytical benchmarking while quantifying generalization error, residual behaviour, and feature reliance.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. What baseline must the model beat?
2. Which preprocessing is learned?
3. Which algorithm generalizes best?
4. Where are errors largest?

        ## Definition of done

        - [ ] Leakage-safe split
- [ ] Dummy baseline
- [ ] Linear and tree candidates
- [ ] Cross-validation
- [ ] Holdout metrics
- [ ] Residual/slice analysis
- [ ] Saved pipeline

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Random split unlike future deployment | Use temporal/geographic split when such fields exist. |
| Price data selection bias | Limit claims to represented properties. |
| Importance read as causality | Describe predictive reliance only. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Load and split

Reserve a test set before model selection. The supplied data has no date, so this is an illustrative random holdout.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

df=pd.read_csv(DATA_DIR/"house_prices.csv")
X=df.drop(columns="price"); y=df.price
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=.2,random_state=42)
print(Xdev.shape,Xtest.shape,y.describe())

## 2. Preprocessing and candidates

Fit numeric imputation/scaling and categorical imputation/encoding inside every candidate pipeline.

In [ ]:
num=X.select_dtypes(include="number").columns.tolist()
cat=X.select_dtypes(exclude="number").columns.tolist()
prep=ColumnTransformer([
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                     ("encode",OneHotEncoder(handle_unknown="ignore"))]),cat),
])
candidates={
    "dummy":Pipeline([("prep",prep),("model",DummyRegressor(strategy="median"))]),
    "ridge":Pipeline([("prep",prep),("model",Ridge(alpha=5))]),
    "forest":Pipeline([("prep",prep),("model",RandomForestRegressor(
        n_estimators=220,min_samples_leaf=3,max_features=.8,random_state=42,n_jobs=1))]),
}

## 3. Cross-validation

Use identical folds and multiple metrics. The test set remains untouched.

In [ ]:
cv=KFold(5,shuffle=True,random_state=42)
rows=[]
for name,model in candidates.items():
    result=cross_validate(model,Xdev,ydev,cv=cv,
                          scoring={"mae":"neg_mean_absolute_error","r2":"r2"},n_jobs=1)
    rows.append({
        "model":name,
        "cv_mae":-result["test_mae"].mean(),
        "cv_mae_sd":result["test_mae"].std(),
        "cv_r2":result["test_r2"].mean(),
    })
cv_results=pd.DataFrame(rows).sort_values("cv_mae")
display(cv_results)

## 4. Final holdout evaluation

Fit the selected candidate once on development data and evaluate on the untouched holdout.

In [ ]:
selected_name=cv_results.iloc[0]["model"]
selected=candidates[selected_name].fit(Xdev,ydev)
pred=selected.predict(Xtest)
metrics={
    "model":selected_name,
    "MAE":mean_absolute_error(ytest,pred),
    "RMSE":mean_squared_error(ytest,pred)**.5,
    "R2":r2_score(ytest,pred),
}
print(metrics)

## 5. Residual and slice analysis

Inspect error magnitude across price bands and avoid relying on one aggregate score.

In [ ]:
results=Xtest.copy()
results["actual"]=ytest.to_numpy()
results["prediction"]=pred
results["residual"]=results.actual-results.prediction
results["absolute_error"]=results.residual.abs()
results["price_band"]=pd.qcut(results.actual,4,duplicates="drop")
display(results.groupby("price_band",observed=True).absolute_error.agg(["count","mean","median"]).round(2))

fig,ax=plt.subplots(figsize=(6,4))
ax.scatter(results.prediction,results.residual,alpha=.45)
ax.axhline(0,linewidth=1)
ax.set(title="Residuals versus predictions",xlabel="Predicted price",ylabel="Actual − predicted")
plt.show()

## 6. Artifact and report

Persist the whole pipeline and a machine-readable metrics record.

In [ ]:
artifact=ARTIFACT_DIR/"capstone_house_price_pipeline.joblib"
joblib.dump(selected,artifact)
report={"metrics":{k:(float(v) if isinstance(v,(np.floating,float)) else v) for k,v in metrics.items()},
        "features":list(X.columns),"split":"random 80/20, seed 42",
        "excluded_use":"Not a lending, taxation, or individual valuation system."}
(ARTIFACT_DIR/"capstone_house_price_report.json").write_text(json.dumps(report,indent=2))
print(artifact)

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.